## **CREATE DATAHUB DOMAIN**

In [1]:
# THIS WILL BE USEFULL IF TEXT TO SQL SYSTEM IMPROVE TO ACCOMODATE SEVERAL DATABASES; CURRENTLY FOR THIS THESIS SCOPE OF WORK 
# DOMAIN WILL NOT BE USED BECAUSE ONLY USE CLASSICMODELS MYSQL DATABASE.
import os
import requests
from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv(), override=False)

GMS = os.getenv("DATAHUB_GMS_URL", "http://datahub-gms:8080")
TOKEN = os.getenv("DATAHUB_TOKEN")

HEAD = {"Content-Type": "application/json"}
if TOKEN:
    HEAD["Authorization"] = f"Bearer {TOKEN}"
else:
    HEAD["X-DataHub-Actor"] = "urn:li:corpuser:admin"

# list domain yang mau kamu punya di DataHub
DOMAINS = [
    {
        "id": "sales",
        "name": "Sales",
        "description": "Sales-related datasets",
    },
    {
        "id": "crm",
        "name": "CRM",
        "description": "Customer / CRM datasets",
    },
    {
        "id": "hr",
        "name": "HR",
        "description": "Human Resources datasets",
    },
    {
        "id": "catalog",
        "name": "Catalog",
        "description": "Product / catalog datasets",
    },
    {
        "id": "finance",
        "name": "Finance",
        "description": "Finance-related datasets",
    },
    {
        "id": "geo",
        "name": "Geo",
        "description": "Geographical / office datasets",
    },
]

MUTATION = """
mutation createDomain($input: CreateDomainInput!) {
  createDomain(input: $input)
}
"""

def create_domain(dom):
    payload = {
        "query": MUTATION,
        "variables": {
            "input": {
                "id": dom["id"],
                "name": dom["name"],
                "description": dom.get("description", ""),
            }
        },
    }
    r = requests.post(f"{GMS}/api/graphql", headers=HEAD, json=payload, timeout=30)
    r.raise_for_status()
    js = r.json()
    if "errors" in js:
        msg = js["errors"][0].get("message", "")
        print(f"[SKIP] {dom['id']} -> {msg}")
    else:
        print(f"[OK]   {dom['id']} created")

def main():
    print(f"Creating {len(DOMAINS)} domains to {GMS} ...")
    for d in DOMAINS:
        try:
            create_domain(d)
        except Exception as e:
            print(f"[ERR]  {d['id']} -> {e}")

if __name__ == "__main__":
    main()

Creating 6 domains to http://datahub-gms:8080 ...
[ERR]  sales -> 401 Client Error: Unauthorized for url: http://datahub-gms:8080/api/graphql
[ERR]  crm -> 401 Client Error: Unauthorized for url: http://datahub-gms:8080/api/graphql
[ERR]  hr -> 401 Client Error: Unauthorized for url: http://datahub-gms:8080/api/graphql
[ERR]  catalog -> 401 Client Error: Unauthorized for url: http://datahub-gms:8080/api/graphql
[ERR]  finance -> 401 Client Error: Unauthorized for url: http://datahub-gms:8080/api/graphql
[ERR]  geo -> 401 Client Error: Unauthorized for url: http://datahub-gms:8080/api/graphql


Setelah buat domain, baru jalankan ingestion pipeline agar bisa add datasets dengan domain yg telah dibuat.

## **CREATE DATAHUB TAGS FOR DATASET**

In [14]:
#!/usr/bin/env python3
import os
import re
import time
from typing import Any, Dict, List, Set

import requests
from dotenv import find_dotenv, load_dotenv

load_dotenv(find_dotenv(), override=False)

# =========================
# CONFIG
# =========================
DATAHUB_GRAPHQL = os.getenv("DATAHUB_GRAPHQL_URL", "http://datahub-gms:8080/api/graphql")
DATAHUB_TOKEN = os.getenv("DATAHUB_TOKEN")
TARGET_ENUM_TYPE = "DATASET_FIELD"

DATASET_URNS: Dict[str, str] = {
    "customers": "urn:li:dataset:(urn:li:dataPlatform:mysql,classicmodels.customers,PROD)",
    "employees": "urn:li:dataset:(urn:li:dataPlatform:mysql,classicmodels.employees,PROD)",
    "products": "urn:li:dataset:(urn:li:dataPlatform:mysql,classicmodels.products,PROD)",
    "productlines": "urn:li:dataset:(urn:li:dataPlatform:mysql,classicmodels.productlines,PROD)",
    "orders": "urn:li:dataset:(urn:li:dataPlatform:mysql,classicmodels.orders,PROD)",
    "orderdetails": "urn:li:dataset:(urn:li:dataPlatform:mysql,classicmodels.orderdetails,PROD)",
    "payments": "urn:li:dataset:(urn:li:dataPlatform:mysql,classicmodels.payments,PROD)",
    "offices": "urn:li:dataset:(urn:li:dataPlatform:mysql,classicmodels.offices,PROD)",
}

ALIAS_MAPPING = {
    "customers": {
        "customerNumber": {"aliases": ["client id", "account id", "customer code"], "role": "PK"},
        "customerName": {"aliases": ["client name", "company name", "purchaser"], "role": "Dimension"},
        "contactLastName": {"aliases": ["surname", "contact person last name"], "role": "Dimension"},
        "contactFirstName": {"aliases": ["given name", "contact person first name"], "role": "Dimension"},
        "phone": {"aliases": ["contact number", "mobile", "telephone"], "role": "Contact"},
        "addressLine1": {"aliases": ["street address", "primary address"], "role": "Address"},
        "addressLine2": {"aliases": ["suite", "floor", "unit"], "role": "Address"},
        "city": {"aliases": ["municipality", "town"], "role": "Geo"},
        "state": {"aliases": ["region", "province"], "role": "Geo"},
        "postalCode": {"aliases": ["zip code", "post code"], "role": "Geo"},
        "country": {"aliases": ["nation"], "role": "Geo"},
        "salesRepEmployeeNumber": {"aliases": ["account manager id", "sales rep id", "assigned sales person"], "role": "FK"},
        "creditLimit": {"aliases": ["credit ceiling", "max credit", "spending limit"], "role": "Metric"},
    },
    "employees": {
        "employeeNumber": {"aliases": ["staff id", "worker id"], "role": "PK"},
        "lastName": {"aliases": ["surname"], "role": "Dimension"},
        "firstName": {"aliases": ["given name"], "role": "Dimension"},
        "extension": {"aliases": ["internal phone", "ext"], "role": "Contact"},
        "email": {"aliases": ["work email", "contact email"], "role": "Contact"},
        "officeCode": {"aliases": ["office id", "location code"], "role": "FK"},
        "reportsTo": {"aliases": ["manager id", "supervisor id", "direct report"], "role": "FK"},
        "jobTitle": {"aliases": ["position", "designation", "role"], "role": "Dimension"},
    },
    "offices": {
        "officeCode": {"aliases": ["office id", "branch code"], "role": "PK"},
        "city": {"aliases": ["office city", "branch location"], "role": "Geo"},
        "phone": {"aliases": ["office phone"], "role": "Contact"},
        "addressLine1": {"aliases": ["street"], "role": "Address"},
        "addressLine2": {"aliases": ["suite"], "role": "Address"},
        "state": {"aliases": ["region"], "role": "Geo"},
        "country": {"aliases": ["nation"], "role": "Geo"},
        "postalCode": {"aliases": ["zip"], "role": "Geo"},
        "territory": {"aliases": ["sales region", "area"], "role": "Dimension"},
    },
    "orderdetails": {
        "orderNumber": {"aliases": ["order id", "transaction id"], "role": "FK"},
        "productCode": {"aliases": ["sku", "item code"], "role": "FK"},
        "quantityOrdered": {"aliases": ["qty", "count", "units", "volume"], "role": "Metric"},
        "priceEach": {"aliases": ["unit price", "selling price", "deal price"], "role": "Metric"},
        "orderLineNumber": {"aliases": ["line sequence", "item number"], "role": "Dimension"},
    },
    "orders": {
        "orderNumber": {"aliases": ["order id", "invoice id"], "role": "PK"},
        "orderDate": {"aliases": ["purchase date", "transaction date"], "role": "Date"},
        "requiredDate": {"aliases": ["due date", "deadline"], "role": "Date"},
        "shippedDate": {"aliases": ["delivery date", "dispatch date"], "role": "Date"},
        "status": {"aliases": ["order state", "condition"], "role": "Dimension"},
        "comments": {"aliases": ["notes", "instructions"], "role": "Text"},
        "customerNumber": {"aliases": ["buyer id", "purchaser id"], "role": "FK"},
    },
    "payments": {
        "customerNumber": {"aliases": ["payer id"], "role": "FK"},
        "checkNumber": {"aliases": ["payment reference", "transaction ref"], "role": "PK"},
        "paymentDate": {"aliases": ["transaction date"], "role": "Date"},
        "amount": {"aliases": ["payment value", "total paid", "money received"], "role": "Metric"},
    },
    "productlines": {
        "productLine": {"aliases": ["category", "product family"], "role": "PK"},
        "textDescription": {"aliases": ["description", "details"], "role": "Text"},
        "htmlDescription": {"aliases": ["html content"], "role": "Text"},
        "image": {"aliases": ["picture", "photo"], "role": "Binary"},
    },
    "products": {
        "productCode": {"aliases": ["sku", "item id"], "role": "PK"},
        "productName": {"aliases": ["item name", "model name"], "role": "Dimension"},
        "productLine": {"aliases": ["category", "family"], "role": "FK"},
        "productScale": {"aliases": ["model scale", "ratio"], "role": "Dimension"},
        "productVendor": {"aliases": ["manufacturer", "brand", "maker"], "role": "Dimension"},
        "productDescription": {"aliases": ["specs", "details"], "role": "Text"},
        "quantityInStock": {"aliases": ["inventory", "stock level", "available units"], "role": "Metric"},
        "buyPrice": {"aliases": ["cost", "cogs", "purchase price"], "role": "Metric"},
        "MSRP": {"aliases": ["retail price", "list price", "catalog price"], "role": "Metric"},
    },
}

ALIAS_TAG_PREFIX = "alias:"
ROLE_TAG_PREFIX = "role:"

ROLE_CANONICAL = {
    "pk": "pk",
    "primary_key": "primary_key",
    "primary key": "primary_key",
    "fk": "fk",
    "foreign_key": "foreign_key",
    "foreign key": "foreign_key",
    "id": "id",
    "identifier": "identifier",
    "measure": "measure",
    "metric": "measure",
    "dimension": "dimension",
    "date": "date",
    "datetime": "date",
    "time": "date",
    "temporal": "temporal",
    "geo": "geo",
    "contact": "contact",
    "address": "address",
    "text": "text",
    "binary": "binary",
}

# =========================
# GRAPHQL HELPERS
# =========================
def _headers() -> Dict[str, str]:
    headers = {"Content-Type": "application/json"}
    if DATAHUB_TOKEN:
        headers["Authorization"] = f"Bearer {DATAHUB_TOKEN}"
    else:
        headers["X-DataHub-Actor"] = "urn:li:corpuser:admin"
    return headers

def gql(query: str, variables: dict) -> Dict[str, Any]:
    try:
        resp = requests.post(
            DATAHUB_GRAPHQL,
            headers=_headers(),
            json={"query": query, "variables": variables},
            timeout=30,
        )
        resp.raise_for_status()
        return resp.json()
    except Exception as exc:
        return {"errors": [{"message": str(exc)}]}

def get_dataset_fields(urn: str) -> Set[str]:
    query = """
    query getDatasetSchema($urn: String!) {
      dataset(urn: $urn) {
        schemaMetadata { fields { fieldPath } }
      }
    }
    """
    res = gql(query, {"urn": urn})
    fields = res.get("data", {}).get("dataset", {}).get("schemaMetadata", {}).get("fields", [])
    return {f["fieldPath"] for f in fields}

def generate_urn(display_name: str) -> str:
    tag_id = re.sub(
        r"[^A-Za-z0-9._-]",
        "_",
        display_name.replace(":", ".").replace(" ", "_"),
    ).lower().strip("._-")
    return f"urn:li:tag:{tag_id}"

def _normalize_token(text: str) -> str:
    lowered = str(text or "").strip().lower().replace("-", "_").replace(" ", "_")
    lowered = re.sub(r"_+", "_", lowered)
    return lowered.strip("_")

def normalize_role(role: str) -> str:
    key = _normalize_token(role)
    return ROLE_CANONICAL.get(key, ROLE_CANONICAL.get(str(role or "").strip().lower(), key))

def _tokenize_semantic_text(*parts: str) -> Set[str]:
    tokens: Set[str] = set()
    for part in parts:
        raw = str(part or "")
        if not raw:
            continue
        spaced = re.sub(r"([a-z0-9])([A-Z])", r"\1 \2", raw)
        spaced = spaced.replace("_", " ").replace("-", " ")
        for token in re.findall(r"[a-z0-9]+", spaced.lower()):
            if token:
                tokens.add(token)
    return tokens

def _cfg_aliases(cfg: Dict[str, Any]) -> List[str]:
    aliases: List[str] = []
    seen: Set[str] = set()
    for raw_alias in cfg.get("aliases", []) or []:
        alias = str(raw_alias or "").strip()
        if not alias:
            continue
        key = alias.lower()
        if key in seen:
            continue
        seen.add(key)
        aliases.append(alias)
    return aliases

def _infer_base_roles(field_name: str, aliases: List[str]) -> List[str]:
    tokens = _tokenize_semantic_text(field_name, *aliases)
    inferred: List[str] = []

    def add(role: str) -> None:
        normalized = normalize_role(role)
        if normalized and normalized not in inferred:
            inferred.append(normalized)

    if {"date", "time", "year", "month", "day"}.intersection(tokens):
        add("date")
    if {"email", "phone", "fax", "mobile", "telephone", "extension", "ext", "contact"}.intersection(tokens):
        add("contact")
    if {"address", "street", "suite"}.intersection(tokens):
        add("address")
    if {"city", "state", "province", "region", "country", "nation", "postal", "zip", "postcode", "territory"}.intersection(tokens):
        add("geo")
    if {"image", "photo", "picture", "binary"}.intersection(tokens):
        add("binary")
    if {"description", "details", "notes", "instructions", "comment", "comments", "html", "text"}.intersection(tokens):
        add("text")
    if {"id", "code", "number", "reference", "ref", "identifier"}.intersection(tokens):
        add("id")
    if {"amount", "price", "cost", "quantity", "qty", "stock", "inventory", "credit", "limit", "revenue", "sales", "paid", "payment", "units", "volume", "ratio"}.intersection(tokens):
        add("measure")
    if not inferred:
        add("dimension")
    return inferred

def _expanded_roles_for_field(table_name: str, field_name: str, cfg: Dict[str, Any]) -> List[str]:
    aliases = _cfg_aliases(cfg)
    roles: List[str] = []

    def add(role: str) -> None:
        normalized = normalize_role(role)
        if normalized and normalized not in roles:
            roles.append(normalized)

    raw_roles = cfg.get("roles")
    if isinstance(raw_roles, list):
        for raw_role in raw_roles:
            add(str(raw_role or ""))
    elif "role" in cfg:
        add(str(cfg.get("role") or ""))

    for raw_role in cfg.get("extra_roles", []) or []:
        add(str(raw_role or ""))

    if not roles:
        for inferred in _infer_base_roles(field_name, aliases):
            add(inferred)

    tokens = _tokenize_semantic_text(table_name, field_name, *aliases)

    if "pk" in roles:
        add("id")
        add("identifier")
        add("primary_key")
    if "fk" in roles:
        add("id")
        add("identifier")
        add("foreign_key")
        add("reference")
    if "id" in roles:
        add("identifier")

    if "dimension" in roles:
        if {"name", "first", "last", "given", "surname"}.intersection(tokens):
            add("name")
            add("entity_name")
        if "status" in tokens:
            add("status")
        if {"category", "family", "line", "type", "territory", "vendor", "title"}.intersection(tokens):
            add("classification")

    if "date" in roles:
        add("temporal")
        if {"order", "purchase", "payment", "transaction", "invoice"}.intersection(tokens):
            add("transaction_date")
        if {"required", "due", "deadline"}.intersection(tokens):
            add("due_date")
        if {"shipped", "delivery", "dispatch"}.intersection(tokens):
            add("fulfillment_date")

    if "measure" in roles:
        add("metric")
        if {"amount"}.intersection(tokens):
            add("amount")
        if {"price", "cost", "msrp"}.intersection(tokens):
            add("price")
        if {"amount", "price", "cost", "sales", "revenue", "profit", "credit", "debit", "money", "paid", "payment", "msrp"}.intersection(tokens):
            add("monetary")
        if {"quantity", "qty", "units", "count", "volume", "stock", "inventory"}.intersection(tokens):
            add("quantity")
        if {"stock", "inventory", "available"}.intersection(tokens):
            add("inventory")
        if {"order", "ordered", "transaction", "sales", "sold"}.intersection(tokens) and {"quantity", "qty", "units", "count", "volume"}.intersection(tokens):
            add("transaction_quantity")
        if {"payment", "paid", "transaction"}.intersection(tokens) and {"amount", "money", "value", "total"}.intersection(tokens):
            add("transaction_amount")
        if "limit" in tokens:
            add("threshold")

    if "contact" in roles:
        if {"email", "mail"}.intersection(tokens):
            add("email")
        if {"phone", "fax", "extension", "ext", "mobile", "telephone"}.intersection(tokens):
            add("phone")

    if "address" in roles:
        add("location_text")

    if "geo" in roles:
        if "country" in tokens or "nation" in tokens:
            add("country")
        if {"state", "province", "region", "territory"}.intersection(tokens):
            add("region")
        if {"city", "town", "municipality"}.intersection(tokens):
            add("city")
        if {"postal", "zip", "postcode"}.intersection(tokens):
            add("postal_code")

    if "text" in roles and {"description", "details", "notes", "instructions", "comments", "comment"}.intersection(tokens):
        add("narrative_text")

    if "binary" in roles and {"image", "picture", "photo"}.intersection(tokens):
        add("media")

    return roles

def create_tag(display_name: str) -> None:
    tag_id = re.sub(
        r"[^A-Za-z0-9._-]",
        "_",
        display_name.replace(":", ".").replace(" ", "_"),
    ).lower().strip("._-")
    res = gql(
        "mutation c($i: CreateTagInput!){createTag(input:$i)}",
        {"i": {"id": tag_id, "name": display_name}},
    )
    if "errors" in res:
        print(f"[FAIL] create_tag {display_name}: {res['errors'][0]['message']}")

def attach_tags(dataset_urn: str, field_path: str, tag_urns: List[str]) -> None:
    variables = {
        "i": {
            "resourceUrn": dataset_urn,
            "tagUrns": tag_urns,
            "subResource": field_path,
            "subResourceType": TARGET_ENUM_TYPE,
        }
    }
    res = gql("""mutation add($i: AddTagsInput!){addTags(input:$i)}""", variables)
    if "errors" in res:
        print(f"[FAIL] {field_path}: {res['errors'][0]['message']}")
    else:
        print(f"[OK] {field_path}")

def main() -> None:
    print("[INFO] MEMULAI ENRICHMENT 2-FASE (PROVISIONING -> LINKING)...")

    all_unique_tags: Set[str] = set()
    print("\n--- FASE 1: Mengumpulkan & Membuat Tag ---")

    for table_name, fields in ALIAS_MAPPING.items():
        for field_name, cfg in fields.items():
            for alias in _cfg_aliases(cfg):
                all_unique_tags.add(f"{ALIAS_TAG_PREFIX}{alias}")
            for role in _expanded_roles_for_field(table_name, field_name, cfg):
                all_unique_tags.add(f"{ROLE_TAG_PREFIX}{role}")

    print(f"[INFO] Ditemukan {len(all_unique_tags)} tag unik. Membuat tag ke DataHub...")
    for idx, tag_label in enumerate(sorted(all_unique_tags), 1):
        create_tag(tag_label)
        if idx % 10 == 1 or idx == len(all_unique_tags):
            print(f"   ... dibuat {idx} tag")

    settle_time = 15
    print(f"\n--- FASE 2: Waiting for Indexing ({settle_time}s) ---")
    print("Menunggu Kafka & Elasticsearch sinkronisasi agar tidak terjadi 'Urn does not exist'...")
    time.sleep(settle_time)
    print("Selesai menunggu.")

    print("\n--- FASE 3: Menempelkan Tag ke Kolom ---")
    for dataset_name, mapping_fields in ALIAS_MAPPING.items():
        if dataset_name not in DATASET_URNS:
            continue
        dataset_urn = DATASET_URNS[dataset_name]

        try:
            actual_fields = get_dataset_fields(dataset_urn)
            if not actual_fields:
                continue
        except Exception:
            continue

        for target_field, cfg in mapping_fields.items():
            real_field_path = next((f for f in actual_fields if f.lower() == target_field.lower()), None)
            if not real_field_path:
                print(f"[SKIP] {dataset_name}: Field '{target_field}' tidak ditemukan.")
                continue

            tag_urns: List[str] = []
            for alias in _cfg_aliases(cfg):
                tag_urns.append(generate_urn(f"{ALIAS_TAG_PREFIX}{alias}"))
            for role in _expanded_roles_for_field(dataset_name, target_field, cfg):
                tag_urns.append(generate_urn(f"{ROLE_TAG_PREFIX}{role}"))

            attach_tags(dataset_urn, real_field_path, tag_urns)

if __name__ == "__main__":
    main()


[INFO] MEMULAI ENRICHMENT 2-FASE (PROVISIONING -> LINKING)...

--- FASE 1: Mengumpulkan & Membuat Tag ---
[INFO] Ditemukan 151 tag unik. Membuat tag ke DataHub...
   ... dibuat 1 tag
   ... dibuat 11 tag
   ... dibuat 21 tag
   ... dibuat 31 tag
   ... dibuat 41 tag
   ... dibuat 51 tag
   ... dibuat 61 tag
   ... dibuat 71 tag
   ... dibuat 81 tag
   ... dibuat 91 tag
   ... dibuat 101 tag
   ... dibuat 111 tag
   ... dibuat 121 tag
   ... dibuat 131 tag
   ... dibuat 141 tag
   ... dibuat 151 tag

--- FASE 2: Waiting for Indexing (15s) ---
Menunggu Kafka & Elasticsearch sinkronisasi agar tidak terjadi 'Urn does not exist'...
Selesai menunggu.

--- FASE 3: Menempelkan Tag ke Kolom ---
[OK] customerNumber
[OK] customerName
[OK] contactLastName
[OK] contactFirstName
[OK] phone
[OK] addressLine1
[OK] addressLine2
[OK] city
[OK] state
[OK] postalCode
[OK] country
[OK] salesRepEmployeeNumber
[OK] creditLimit
[OK] employeeNumber
[OK] lastName
[OK] firstName
[OK] extension
[OK] email
[OK] of

In [3]:
#!/usr/bin/env python3
import os
import time
import requests
from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv(), override=False)

GMS   = os.environ.get("DATAHUB_GMS_URL", "http://datahub-gms:8080")
TOKEN = os.environ.get("DATAHUB_TOKEN")
HEAD  = {"Content-Type": "application/json"}
if TOKEN:
    HEAD["Authorization"] = f"Bearer {TOKEN}"
else:
    HEAD["X-DataHub-Actor"] = "urn:li:corpuser:admin"

# --- di output-mu ketemu pola: alias.xxx (di urn/id) dan alias:xxx (di name)
# jadi kita bikin prefix yang "bebas pemisah"
BASE_PREFIXES = ("alias", "role")

DRY_RUN  = True
PAGE_SZ  = 200

def gql(q, v):
    r = requests.post(
        f"{GMS}/api/graphql",
        headers=HEAD,
        json={"query": q, "variables": v},
        timeout=30,
    )
    r.raise_for_status()
    js = r.json()
    if "errors" in js:
        raise RuntimeError(js["errors"])
    return js["data"]

def list_all_tags():
    """Return list of (urn, id, name) for all tags."""
    out, start = [], 0
    q = """
    query($start:Int!, $count:Int!) {
      search(input:{type:TAG, query:"", start:$start, count:$count}) {
        total
        searchResults {
          entity {
            urn
            ... on Tag {
              urn
              properties { name }
            }
          }
        }
      }
    }"""
    while True:
        d = gql(q, {"start": start, "count": PAGE_SZ})
        sr = d["search"]["searchResults"]
        if not sr:
            break
        for r in sr:
            ent = r["entity"]
            urn = ent["urn"]
            # di kasusmu: urn:li:tag:alias.address_1
            tag_id = urn.split("urn:li:tag:")[-1]
            props = ent.get("properties") or {}
            name  = props.get("name")  # contoh: "alias:address 1"
            out.append((urn, tag_id, name))
        start += PAGE_SZ
        if start >= d["search"]["total"]:
            break
    return out

def hard_delete(urn):
    r = requests.post(
        f"{GMS}/entities?action=delete",
        headers=HEAD,
        json={"urn": urn, "hardDelete": True},
        timeout=30,
    )
    r.raise_for_status()
    return r.json()

def match_prefix(tag_id: str, name: str) -> bool:
    """
    Kembalikan True kalau tag ini masuk salah satu BASE_PREFIXES.
    Kita normalize:
      - lowercase
      - ganti ":" jadi "."
    lalu cek prefix.
    """
    candidates = []

    if tag_id:
        candidates.append(tag_id.lower())
        candidates.append(tag_id.lower().replace(":", "."))
    if name:
        candidates.append(name.lower())
        candidates.append(name.lower().replace(":", "."))

    for cand in candidates:
        for base in BASE_PREFIXES:
            if cand.startswith(base + "."):
                return True
    return False

def main():
    all_tags = list_all_tags()

    print("== tags ditemukan dari GMS ==")
    for urn, tid, name in all_tags:
        print(f"- {urn}  (id={tid}, name={name})")

    # filter
    targets = []
    for urn, tid, name in all_tags:
        if match_prefix(tid, name):
            targets.append((urn, tid, name))

    if not targets:
        # kasih info prefix yang sekarang dipakai
        expanded = [f"{p}. / {p}:" for p in BASE_PREFIXES]
        print("\nNo matching tags with prefixes (normalized):", ", ".join(expanded))
        return

    print(f"\nFound {len(targets)} tag(s) to remove:")
    for urn, tid, name in targets:
        print(f"  - {urn}  (name={name}, id={tid})")

    if DRY_RUN:
        print("\n[DRY RUN] Set DRY_RUN=False to actually delete.")
        return

    print("\nDeleting...")
    ok = 0
    for urn, tid, name in targets:
        try:
            hard_delete(urn)
            ok += 1
            print(f"  OK  {urn}  ({name})")
            time.sleep(0.05)
        except Exception as e:
            print(f"  FAIL {urn} -> {e}")

    print(f"\nDone. Deleted {ok}/{len(targets)} tags.")

if __name__ == "__main__":
    main()

HTTPError: 401 Client Error: Unauthorized for url: http://datahub-gms:8080/api/graphql

# **DEATTACH TAGS FROM DATASET**

In [6]:
from collections import defaultdict
from typing import Any, Dict, List, Set
# run this first, or make sure this already exists above

import os
import requests
from dotenv import find_dotenv, load_dotenv

load_dotenv(find_dotenv(), override=False)

DATAHUB_GRAPHQL = os.getenv("DATAHUB_GRAPHQL_URL", "http://datahub-gms:8080/api/graphql")
DATAHUB_TOKEN = os.getenv("DATAHUB_TOKEN")
TARGET_ENUM_TYPE = "DATASET_FIELD"

def _headers():
    headers = {"Content-Type": "application/json"}
    if DATAHUB_TOKEN:
        headers["Authorization"] = f"Bearer {DATAHUB_TOKEN}"
    else:
        headers["X-DataHub-Actor"] = "urn:li:corpuser:admin"
    return headers

def gql(query: str, variables: dict):
    try:
        resp = requests.post(
            DATAHUB_GRAPHQL,
            headers=_headers(),
            json={"query": query, "variables": variables},
            timeout=30,
        )
        resp.raise_for_status()
        return resp.json()
    except Exception as exc:
        return {"errors": [{"message": str(exc)}]}
        
DATASET_URNS: Dict[str, str] = {
    "customers": "urn:li:dataset:(urn:li:dataPlatform:mysql,classicmodels.customers,PROD)",
    "employees": "urn:li:dataset:(urn:li:dataPlatform:mysql,classicmodels.employees,PROD)",
    "products": "urn:li:dataset:(urn:li:dataPlatform:mysql,classicmodels.products,PROD)",
    "productlines": "urn:li:dataset:(urn:li:dataPlatform:mysql,classicmodels.productlines,PROD)",
    "orders": "urn:li:dataset:(urn:li:dataPlatform:mysql,classicmodels.orders,PROD)",
    "orderdetails": "urn:li:dataset:(urn:li:dataPlatform:mysql,classicmodels.orderdetails,PROD)",
    "payments": "urn:li:dataset:(urn:li:dataPlatform:mysql,classicmodels.payments,PROD)",
    "offices": "urn:li:dataset:(urn:li:dataPlatform:mysql,classicmodels.offices,PROD)",
}

FIELD_TAGS_QUERY = """
query datasetFieldTags($urn: String!) {
  dataset(urn: $urn) {
    urn
    schemaMetadata(version: 0) {
      fields {
        fieldPath
        globalTags {
          tags {
            tag {
              urn
              properties { name }
            }
          }
        }
      }
    }
    editableSchemaMetadata {
      editableSchemaFieldInfo {
        fieldPath
        globalTags {
          tags {
            tag {
              urn
              properties { name }
            }
          }
        }
      }
    }
  }
}
""".strip()

REMOVE_TAG_MUTATION = """
mutation remove($input: TagAssociationInput!) {
  removeTag(input: $input)
}
""".strip()

def _extract_tag_urns(tag_assoc_list):
    out = []
    for item in tag_assoc_list or []:
        tag = (item or {}).get("tag") or {}
        urn = tag.get("urn")
        if urn:
            out.append(urn)
    return out

def get_field_tag_map(dataset_urn: str):
    res = gql(FIELD_TAGS_QUERY, {"urn": dataset_urn})
    ds = ((res or {}).get("data") or {}).get("dataset") or {}
    out = defaultdict(set)

    for f in ((ds.get("schemaMetadata") or {}).get("fields") or []):
        fp = f.get("fieldPath")
        if fp:
            out[fp].update(_extract_tag_urns(((f.get("globalTags") or {}).get("tags") or [])))

    for f in ((ds.get("editableSchemaMetadata") or {}).get("editableSchemaFieldInfo") or []):
        fp = f.get("fieldPath")
        if fp:
            out[fp].update(_extract_tag_urns(((f.get("globalTags") or {}).get("tags") or [])))

    return {fp: sorted(tag_urns) for fp, tag_urns in out.items()}

def remove_field_tag(dataset_urn: str, field_path: str, tag_urn: str):
    variables = {
        "input": {
            "resourceUrn": dataset_urn,
            "tagUrn": tag_urn,
            "subResource": field_path,
            "subResourceType": TARGET_ENUM_TYPE,
        }
    }
    res = gql(REMOVE_TAG_MUTATION, variables)
    if "errors" in res:
        print(f"[FAIL] {field_path} :: {tag_urn} -> {res['errors'][0]['message']}")
    else:
        print(f"[OK] removed {tag_urn} from {field_path}")

def remove_field_tags_for_dataset(dataset_urn: str, only_t2sql_tags: bool = True):
    # Your notebook-generated tag URNs become like:
    #   urn:li:tag:alias.client_name
    #   urn:li:tag:role.measure
    owned_prefixes = ("urn:li:tag:alias.", "urn:li:tag:role.")
    tag_map = get_field_tag_map(dataset_urn)

    for field_path, tag_urns in sorted(tag_map.items()):
        for tag_urn in tag_urns:
            if only_t2sql_tags and not tag_urn.startswith(owned_prefixes):
                continue
            remove_field_tag(dataset_urn, field_path, tag_urn)

In [7]:
for dataset_name, dataset_urn in DATASET_URNS.items():
    print(f"\n=== CLEANUP {dataset_name} ===")
    remove_field_tags_for_dataset(dataset_urn, only_t2sql_tags=False)


=== CLEANUP customers ===
[OK] removed urn:li:tag:alias.primary_address from addressLine1
[OK] removed urn:li:tag:alias.street_address from addressLine1
[OK] removed urn:li:tag:role.address from addressLine1
[OK] removed urn:li:tag:alias.floor from addressLine2
[OK] removed urn:li:tag:alias.suite from addressLine2
[OK] removed urn:li:tag:alias.unit from addressLine2
[OK] removed urn:li:tag:role.address from addressLine2
[OK] removed urn:li:tag:alias.municipality from city
[OK] removed urn:li:tag:alias.town from city
[OK] removed urn:li:tag:role.geo from city
[OK] removed urn:li:tag:alias.contact_person_first_name from contactFirstName
[OK] removed urn:li:tag:alias.given_name from contactFirstName
[OK] removed urn:li:tag:role.dimension from contactFirstName
[OK] removed urn:li:tag:alias.contact_person_last_name from contactLastName
[OK] removed urn:li:tag:alias.surname from contactLastName
[OK] removed urn:li:tag:role.dimension from contactLastName
[OK] removed urn:li:tag:alias.nation 

# **DELETE TAGS FROM DATAHUB**

In [12]:
# ============================================
# DELETE ORPHANED T2SQL TAG DEFINITIONS
# - Deletes tag entities like alias:* and role:*
# - Use after field tags have already been detached
# ============================================

import os
from typing import Any, Dict, List

import requests
from dotenv import find_dotenv, load_dotenv

load_dotenv(find_dotenv(), override=False)

DATAHUB_GRAPHQL = os.getenv("DATAHUB_GRAPHQL_URL", "http://datahub-gms:8080/api/graphql")
DATAHUB_TOKEN = os.getenv("DATAHUB_TOKEN")

OWNED_TAG_NAME_PREFIXES = ("alias:", "role:")
OWNED_TAG_URN_PREFIXES = ("urn:li:tag:alias.", "urn:li:tag:role.")

def _headers() -> Dict[str, str]:
    headers = {"Content-Type": "application/json"}
    if DATAHUB_TOKEN:
        headers["Authorization"] = f"Bearer {DATAHUB_TOKEN}"
    else:
        headers["X-DataHub-Actor"] = "urn:li:corpuser:admin"
    return headers

def gql(query: str, variables: dict | None = None) -> Dict[str, Any]:
    try:
        resp = requests.post(
            DATAHUB_GRAPHQL,
            headers=_headers(),
            json={"query": query, "variables": variables or {}},
            timeout=30,
        )
        resp.raise_for_status()
        return resp.json()
    except Exception as exc:
        return {"errors": [{"message": str(exc)}]}

SEARCH_TAGS_QUERY = """
query searchTags($query: String!, $start: Int!, $count: Int!) {
  search(input: { type: TAG, query: $query, start: $start, count: $count }) {
    start
    count
    total
    searchResults {
      entity {
        urn
        type
        ... on Tag {
          properties {
            name
          }
        }
      }
    }
  }
}
""".strip()

DELETE_TAG_MUTATION = """
mutation deleteTag($urn: String!) {
  deleteTag(urn: $urn)
}
""".strip()

def search_tags(query_text: str, start: int = 0, count: int = 100) -> Dict[str, Any]:
    res = gql(SEARCH_TAGS_QUERY, {"query": query_text, "start": start, "count": count})
    if "errors" in res:
        raise RuntimeError(res["errors"][0]["message"])
    return ((res.get("data") or {}).get("search") or {})

def collect_owned_tag_definitions() -> List[Dict[str, str]]:
    found: Dict[str, Dict[str, str]] = {}

    for query_text in OWNED_TAG_NAME_PREFIXES:
        start = 0
        count = 100

        while True:
            block = search_tags(query_text, start=start, count=count)
            results = block.get("searchResults") or []

            for row in results:
                entity = (row or {}).get("entity") or {}
                urn = str(entity.get("urn") or "").strip()
                props = entity.get("properties") or {}
                name = str(props.get("name") or "").strip()

                if not urn:
                    continue

                if urn.startswith(OWNED_TAG_URN_PREFIXES) or name.startswith(OWNED_TAG_NAME_PREFIXES):
                    found[urn] = {"urn": urn, "name": name}

            total = int(block.get("total") or 0)
            start += count
            if start >= total or not results:
                break

    return sorted(found.values(), key=lambda x: (x["name"], x["urn"]))

def delete_tag_definition(tag_urn: str) -> bool:
    res = gql(DELETE_TAG_MUTATION, {"urn": tag_urn})
    if "errors" in res:
        print(f"[FAIL] delete {tag_urn} -> {res['errors'][0]['message']}")
        return False

    ok = bool(((res.get("data") or {}).get("deleteTag")))
    if ok:
        print(f"[OK] deleted {tag_urn}")
    else:
        print(f"[WARN] deleteTag returned falsy for {tag_urn}")
    return ok

def delete_all_owned_tag_definitions() -> None:
    tags = collect_owned_tag_definitions()
    print(f"[INFO] Found {len(tags)} owned tag definitions")

    for tag in tags:
        print(f"[INFO] deleting {tag['name']} :: {tag['urn']}")
        delete_tag_definition(tag["urn"])

    print("[INFO] done")

In [13]:
delete_all_owned_tag_definitions()

[INFO] Found 121 owned tag definitions
[INFO] deleting alias:account id :: urn:li:tag:alias.account_id
[OK] deleted urn:li:tag:alias.account_id
[INFO] deleting alias:account manager id :: urn:li:tag:alias.account_manager_id
[OK] deleted urn:li:tag:alias.account_manager_id
[INFO] deleting alias:area :: urn:li:tag:alias.area
[OK] deleted urn:li:tag:alias.area
[INFO] deleting alias:assigned sales person :: urn:li:tag:alias.assigned_sales_person
[OK] deleted urn:li:tag:alias.assigned_sales_person
[INFO] deleting alias:available units :: urn:li:tag:alias.available_units
[OK] deleted urn:li:tag:alias.available_units
[INFO] deleting alias:branch code :: urn:li:tag:alias.branch_code
[OK] deleted urn:li:tag:alias.branch_code
[INFO] deleting alias:branch location :: urn:li:tag:alias.branch_location
[OK] deleted urn:li:tag:alias.branch_location
[INFO] deleting alias:brand :: urn:li:tag:alias.brand
[OK] deleted urn:li:tag:alias.brand
[INFO] deleting alias:buyer id :: urn:li:tag:alias.buyer_id
[OK]